# DAY 2 — Data Science: Analisis Penjualan UMKM
### SmartData Platform · Fase Data Foundation

Notebook ini membersihkan dan menganalisis data penjualan UMKM fiktif sebagai fondasi data untuk SmartData Platform.

**Memenuhi requirement dari DAY 1:**
- `FR-01` — memuat dataset dari CSV
- `FR-02` — membersihkan & meringkas data

**Alur:** Load → Clean → Analyze → Visualize → Export


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json

plt.style.use("seaborn-v0_8-darkgrid")

## 1. Load — Memuat Data Mentah

Data mentah sengaja dibuat "kotor" untuk mendemonstrasikan proses pembersihan: format tanggal beragam, nama kota tidak konsisten, nilai jumlah yang hilang, dan baris duplikat.

In [ ]:
df = pd.read_csv("data/sales_raw.csv")
print(f"Jumlah baris awal: {len(df)}")
print(f"\nNilai kosong per kolom:")
print(df.isnull().sum())
df.head()

## 2. Clean — Membersihkan Data

Lima langkah pembersihan: hapus duplikat, normalkan nama kota, samakan format tanggal, isi nilai jumlah yang hilang dengan median, dan buat kolom turunan `pendapatan`.

In [ ]:
# 2a. Hapus duplikat
sebelum = len(df)
df = df.drop_duplicates()
print(f"Duplikat dihapus: {sebelum - len(df)} baris")

# 2b. Normalkan nama kota
df["kota"] = df["kota"].str.strip().str.title()

# 2c. Samakan format tanggal
df["tanggal"] = pd.to_datetime(df["tanggal"], format="mixed", dayfirst=True)

# 2d. Isi jumlah yang hilang dengan median
df["jumlah"] = pd.to_numeric(df["jumlah"], errors="coerce")
df["jumlah"] = df["jumlah"].fillna(df["jumlah"].median()).astype(int)

# 2e. Kolom turunan
df["pendapatan"] = df["harga_satuan"] * df["jumlah"]

print(f"Baris bersih akhir: {len(df)}")
df.head()

## 3. Analyze — Ringkasan Insight

In [ ]:
per_produk = df.groupby("produk")["pendapatan"].sum().sort_values(ascending=False)
per_kota = df.groupby("kota")["pendapatan"].sum().sort_values(ascending=False)

print(f"Total pendapatan   : Rp {df['pendapatan'].sum():,.0f}")
print(f"Total transaksi    : {len(df)}")
print(f"Rata-rata/transaksi: Rp {df['pendapatan'].mean():,.0f}")
print(f"Produk terlaris    : {per_produk.index[0]}")
print(f"Kota tertinggi     : {per_kota.index[0]}")

## 4. Visualize — Grafik

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
per_produk.plot(kind="barh", ax=ax, color="#2563eb")
ax.set_title("Pendapatan per Produk", fontsize=14, fontweight="bold")
ax.set_xlabel("Pendapatan (Rp)")
plt.tight_layout()
plt.show()

In [ ]:
per_bulan = df.groupby(df["tanggal"].dt.to_period("M"))["pendapatan"].sum()
fig, ax = plt.subplots(figsize=(10, 5))
per_bulan.plot(kind="line", marker="o", ax=ax, color="#16a34a", linewidth=2)
ax.set_title("Tren Pendapatan Bulanan", fontsize=14, fontweight="bold")
ax.set_ylabel("Pendapatan (Rp)")
plt.tight_layout()
plt.show()

## 5. Export — Simpan untuk DAY 3 (AI)

Ringkasan disimpan ke `output/summary.json`. File ini akan menjadi **input untuk fase AI di DAY 3**, di mana AI akan membaca angka-angka ini dan menghasilkan insight bisnis dalam bahasa natural.

In [ ]:
ringkasan = {
    "total_pendapatan": int(df["pendapatan"].sum()),
    "total_transaksi": len(df),
    "produk_terlaris": per_produk.index[0],
    "kota_tertinggi": per_kota.index[0],
    "pendapatan_per_produk": per_produk.astype(int).to_dict(),
    "pendapatan_per_kota": per_kota.astype(int).to_dict(),
}

with open("output/summary.json", "w", encoding="utf-8") as f:
    json.dump(ringkasan, f, indent=2, ensure_ascii=False)

df.to_csv("data/sales_clean.csv", index=False)
print("Data bersih & ringkasan tersimpan. Siap untuk DAY 3!")